# First-order optimization: fitting a price model with Robbins-Monro

**CAS BMAI HS26, weekend 1, Friday 4 September, the 10:00 coding block.**
Exercise `hs26-bmai-w1-opt-rm-house`, third of three, after Heron's algorithm and
the genie game.

This morning you found a square root by guessing, checking and correcting, and you
satisfied a genie the same way. Both times the answer sat inside an equation. Now
the answer sits inside data, and that is the whole of supervised learning.

**The data is fictitious.** Musterwil is an invented municipality. The twelve houses
were generated by a script with a fixed seed. The prices are invented. Nothing in
this notebook is a fact about Swiss housing.

By the end you should be able to say three sentences out loud.

1. A prediction that is too low means the parameter has to move up, and the size of
   the miss tells you how far.
2. A fixed step can only orbit the answer. A step proportional to the miss can land
   on it.
3. The rule the lecturer boxed on the board is gradient descent on the squared loss,
   using one house at a time. The name for that is stochastic gradient descent.

**How this notebook works.** The cells you fill in are marked
`# COMPLETE THIS LINE:` and carry a `___` where an expression belongs. Each task cell
is followed by a cell that runs your loop on a throwaway problem with a known answer
and prints `PASS` or `FAIL` with a diagnosis. Re-run the pair until it prints `PASS`.


In [ ]:
# @title Setup. Run this cell first, then leave it alone.
# It writes two files next to the notebook and imports them. There is no
# repository to clone, no API key to paste and no network call anywhere in this
# notebook. Everything below runs offline.
NOTEBOOK_VERSION = "v1.0"

# The TA sets this to True only if the block runs short. See the README.
PREFILL_TASK3 = False

HOUSETOOLS_PY = r'''
"""Helper module for hs26-bmai-w1-opt-rm-house.

Loads the twelve fictitious Musterwil houses, draws the three figures, and runs
the four deterministic task checks. Import it, or let the notebook write it to
disk from the copy embedded in its setup cell.

The municipality of Musterwil is invented. The houses are invented. The prices
are invented. Every figure title says so.

Notation, fixed for all three opt- exercises of the 10:00 block and taken from
the annotated deck, handwritten pages 28 to 34:

    x       the feature, living area, in units of 100 square metres
    y       the target, asking price, in thousands of francs
    y_hat   the prediction, w * x + b
    l       the signed miss, y - y_hat
    alpha   the step size
    eps     the stopping tolerance in abs(l) < eps

The step size is called alpha and the tolerance is called eps. The 11:00
gradient descent lecture reuses eps a third time for the width of the finite
difference window, so an exercise must never spend eps on a step size.

Dependencies: numpy for everything, matplotlib for the figures only.
"""
import csv
import os

import numpy as np

# Okabe-Ito, colour blind safe. Red and green are never used as a pair anywhere
# in this course, so the two levels are blue and vermillion, with ink for a
# neutral third and amber reserved for chrome.
BLUE = "#0072B2"
VERMILLION = "#D55E00"
INK = "#1a1a1a"
AMBER = "#B45309"
GREY = "#6b6b6b"

FICTITIOUS = "Musterwil, 12 fictitious houses"

# The lecture house from the annotated deck, handwritten page 29. Raw francs and
# raw square metres, so it reproduces the blackboard exactly.
LECTURE_X = 234.0
LECTURE_Y = 198789.0


# ------------------------------------------------------------------- data ---
def _find_csv(path=None):
    if path is not None:
        return path
    here = os.path.dirname(os.path.abspath(__file__))
    for candidate in ("houses.csv",
                      os.path.join("data", "houses.csv"),
                      os.path.join(here, "houses.csv"),
                      os.path.join(here, "data", "houses.csv"),
                      os.path.join(here, "..", "data", "houses.csv")):
        if os.path.exists(candidate):
            return candidate
    raise IOError("houses.csv was not found. Run data/generate_houses.py, or "
                  "run the setup cell of the notebook, which writes it.")


def load_houses(path=None):
    """Return (area, price, x, y) as float arrays, sorted by area.

    area is square metres and price is francs, as they appear in the CSV.
    x is area / 100 and y is price / 1000, which are the working units of the
    notebook. The scaling is stated in the notebook and it is the reason a
    participant can turn the alpha knob by hand.
    """
    rows = []
    with open(_find_csv(path)) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            rows.append(line)
    reader = csv.DictReader(rows)
    area, price = [], []
    for row in reader:
        area.append(float(row["area_m2"]))
        price.append(float(row["price_chf"]))
    area = np.array(area)
    price = np.array(price)
    order = np.argsort(area, kind="stable")
    area, price = area[order], price[order]
    return area, price, area / 100.0, price / 1000.0


def mse(w, b, x, y):
    """Mean squared error of the line y = w x + b on the twelve houses."""
    return float(np.mean((y - (w * x + b)) ** 2))


def least_squares(x, y):
    """The best line by ordinary least squares. Closed form, no iteration."""
    n = len(x)
    sx, sy = float(np.sum(x)), float(np.sum(y))
    sxx, sxy = float(np.sum(x * x)), float(np.sum(x * y))
    w = (n * sxy - sx * sy) / (n * sxx - sx * sx)
    b = (sy - w * sx) / n
    return w, b


def print_houses(area, price, x, y):
    """Print the twelve houses in both unit systems."""
    print("%s. Nothing below is real market data." % FICTITIOUS)
    print("")
    print("  id   area (m2)    price (CHF)      x = area/100    y = price/1000")
    for i in range(len(area)):
        print("  %2d      %5.0f      %9.0f            %6.2f          %8.1f"
              % (i + 1, area[i], price[i], x[i], y[i]))
    print("")
    print("  mean(x) = %.4f     mean(y) = %.4f" % (x.mean(), y.mean()))


# ---------------------------------------------------------------- figures ---
def _axes(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(labelsize=8, colors=GREY)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GREY)


def plot_houses(x, y):
    """The twelve houses as points, with no line through them yet."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4, 4.0), dpi=110)
    ax.scatter(x, y, s=42, color=BLUE, zorder=3)
    ax.set_xlabel("living area x, in units of 100 square metres", fontsize=9)
    ax.set_ylabel("asking price y, in thousands of francs", fontsize=9)
    ax.set_title("%s. Find the line." % FICTITIOUS, fontsize=10, color=INK)
    ax.grid(True, color="#d8d4ca", linewidth=0.6, zorder=0)
    _axes(ax)
    fig.tight_layout()
    return fig


def plot_orbit(history, x1, y1, title="One house", show=60):
    """Acts 1 and 2. The estimate against the step number, and the miss.

    history is the list of w values a task function returned. The dashed line is
    the exact solution of the one house equation, y1 / x1.

    Only the first `show` steps are drawn. The act 1 run is 400 steps long, and
    painted whole it collapses into a solid band in which the alternation the act
    is about cannot be seen at all. Sixty steps is enough to show the climb and
    then the sawtooth. The label sits in a legend rather than on the axes,
    because on the long run it landed on top of the data.
    """
    import matplotlib.pyplot as plt
    hist_all = np.asarray(history, dtype=float)
    hist = hist_all[:show + 1]
    exact = y1 / x1
    steps = np.arange(len(hist))
    trimmed = len(hist_all) > len(hist)
    note = (" (first %d of %d steps)" % (len(hist) - 1, len(hist_all) - 1)
            if trimmed else "")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.0, 3.4), dpi=110)
    ax1.plot(steps, hist, color=BLUE, linewidth=1.4, marker="o", markersize=2.6,
             label="estimate w")
    ax1.axhline(exact, color=AMBER, linewidth=1.2, linestyle="--",
                label="exact answer y/x = %.4f" % exact)
    ax1.legend(loc="lower right", fontsize=8, frameon=False)
    ax1.set_xlabel("step", fontsize=9)
    ax1.set_ylabel("estimate w", fontsize=9)
    ax1.set_title("%s: the estimate%s" % (title, note), fontsize=10, color=INK)
    _axes(ax1)

    miss = y1 - hist * x1
    ax2.axhline(0.0, color=GREY, linewidth=0.8)
    ax2.plot(steps, miss, color=VERMILLION, linewidth=1.4, marker="o",
             markersize=2.6)
    ax2.set_xlabel("step", fontsize=9)
    ax2.set_ylabel("miss l = y - y_hat", fontsize=9)
    ax2.set_title("%s: the miss%s" % (title, note), fontsize=10, color=INK)
    _axes(ax2)
    fig.suptitle(FICTITIOUS, fontsize=9, color=GREY)
    fig.tight_layout()
    return fig


def plot_fit_grid(history, x, y, picks=(0, 1, 2, 5, 20, -1)):
    """Act 3 and act 4. Six small panels showing the line as it moves.

    history is the list of (w, b) pairs a task function returned. Small
    multiples are used so that nothing depends on an animation writer being
    available in the runtime.
    """
    import matplotlib.pyplot as plt
    hist = [(float(w), float(b)) for w, b in history]
    idx = [i if i >= 0 else len(hist) + i for i in picks]
    idx = [min(i, len(hist) - 1) for i in idx]
    fig, axes = plt.subplots(2, 3, figsize=(9.0, 5.2), dpi=110)
    grid = np.linspace(float(x.min()) * 0.9, float(x.max()) * 1.05, 20)
    for ax, i in zip(axes.ravel(), idx):
        w, b = hist[i]
        ax.scatter(x, y, s=18, color=BLUE, zorder=3)
        ax.plot(grid, w * grid + b, color=VERMILLION, linewidth=1.6, zorder=2)
        ax.set_ylim(0, 2400)
        ax.set_title("step %d\nw = %.1f  b = %.1f  mse = %.0f"
                     % (i, w, b, mse(w, b, x, y)), fontsize=8, color=INK)
        ax.grid(True, color="#d8d4ca", linewidth=0.5, zorder=0)
        _axes(ax)
    fig.suptitle("%s. The line as the loop moves it." % FICTITIOUS,
                 fontsize=10, color=INK)
    fig.tight_layout()
    return fig


def plot_loss_curve(history, x, y, label="mean squared error"):
    """Act 3 and act 4. Mean squared error against step, on a log axis."""
    import matplotlib.pyplot as plt
    losses = [mse(float(w), float(b), x, y) for w, b in history]
    fig, ax = plt.subplots(figsize=(6.6, 3.6), dpi=110)
    ax.plot(np.arange(len(losses)), losses, color=BLUE, linewidth=1.1)
    w_lsq, b_lsq = least_squares(x, y)
    floor = mse(w_lsq, b_lsq, x, y)
    ax.axhline(floor, color=AMBER, linewidth=1.2, linestyle="--")
    ax.annotate("best possible line, mse = %.2f" % floor,
                xy=(len(losses) * 0.35, floor * 1.30), fontsize=8, color=AMBER)
    ax.set_yscale("log")
    ax.set_xlabel("step", fontsize=9)
    ax.set_ylabel(label, fontsize=9)
    ax.set_title("%s. How far the line still is from the houses." % FICTITIOUS,
                 fontsize=10, color=INK)
    _axes(ax)
    fig.tight_layout()
    return fig


def plot_wb_path(history, x, y, every=1):
    """The path of (w, b) over a contour map of the mean squared error.

    This is the surface the 11:00 gradient descent lecture is about to descend.
    """
    import matplotlib.pyplot as plt
    hist = np.asarray([(float(w), float(b)) for w, b in history])
    ws = np.linspace(min(400.0, hist[:, 0].min()), max(1100.0, hist[:, 0].max()), 120)
    bs = np.linspace(min(-50.0, hist[:, 1].min()), max(320.0, hist[:, 1].max()), 120)
    WW, BB = np.meshgrid(ws, bs)
    ZZ = np.zeros_like(WW)
    for i in range(WW.shape[0]):
        for j in range(WW.shape[1]):
            ZZ[i, j] = mse(WW[i, j], BB[i, j], x, y)
    fig, ax = plt.subplots(figsize=(6.6, 4.6), dpi=110)
    levels = np.logspace(np.log10(max(ZZ.min(), 1.0)), np.log10(ZZ.max()), 14)
    cs = ax.contour(WW, BB, ZZ, levels=levels, colors=GREY, linewidths=0.6)
    ax.clabel(cs, inline=True, fontsize=6, fmt="%.0f")
    ax.plot(hist[::every, 0], hist[::every, 1], color=BLUE, linewidth=1.0,
            label="the path the loop took")
    ax.scatter([hist[0, 0]], [hist[0, 1]], s=40, color=INK, zorder=4,
               label="where it started")
    ax.scatter([hist[-1, 0]], [hist[-1, 1]], s=40, color=VERMILLION, zorder=4,
               label="where it ended")
    w_lsq, b_lsq = least_squares(x, y)
    ax.scatter([w_lsq], [b_lsq], s=90, marker="+", color=AMBER, zorder=5,
               linewidths=1.6, label="the best line for this data")
    leg = ax.legend(loc="upper left", fontsize=8, frameon=True)
    # a white patch, because the legend otherwise sits over the contour lines
    leg.get_frame().set_facecolor("white")
    leg.get_frame().set_edgecolor("none")
    ax.set_xlabel("w, the slope", fontsize=9)
    ax.set_ylabel("b, the base", fontsize=9)
    ax.set_title("%s. The loss surface, and the path across it."
                 % FICTITIOUS, fontsize=10, color=INK)
    _axes(ax)
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------- checks ---
def ok(message):
    print("PASS  " + message)
    return True


def fail(message):
    print("FAIL  " + message)
    return False


def _has_blank(fn):
    """True when the participant left a ___ marker in the body of `fn`.

    IPython pre-binds the names _, __ and ___ to the last three results, so a
    leftover ___ evaluates to the empty string instead of raising NameError.
    That produces a baffling message about a str operand. Reading the source is
    the reliable way to catch it, so every check calls this first.
    """
    try:
        import inspect
        src = inspect.getsource(fn)
    except Exception:
        return False
    for line in src.splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            continue
        if "___" in stripped:
            return True
    return False


def filled_in(fn):
    """True when the participant has replaced every ___ in `fn`.

    Cells that are downstream of a task use this so that an unfilled notebook
    prints a sentence instead of a traceback.
    """
    return not _has_blank(fn)


BLANK_MESSAGE = ("There is still a ___ in the cell above. Fill in every line "
                 "marked COMPLETE THIS LINE, then run both cells again.")


def _ref_alternating(areas, prices, w, b, alpha, n_steps, seed=0,
                     recompute=True, x_on_b=False):
    rng = np.random.default_rng(seed)
    history = [(w, b)]
    for _ in range(n_steps):
        i = int(rng.integers(len(areas)))
        x_i, y_i = areas[i], prices[i]
        l = y_i - (w * x_i + b)
        w = w + alpha * l
        if recompute:
            l = y_i - (w * x_i + b)
        b = b + alpha * l * (x_i if x_on_b else 1.0)
        history.append((w, b))
    return history


def _ref_sgd(areas, prices, w, b, alpha, n_epochs, seed=0, decay=None,
             shuffle=True, x_on_w=True, x_on_b=False):
    rng = np.random.default_rng(seed)
    history = [(w, b)]
    t = 0
    for _ in range(n_epochs):
        order = rng.permutation(len(areas)) if shuffle else np.arange(len(areas))
        for i in order:
            x_i, y_i = areas[i], prices[i]
            a_t = alpha if decay is None else alpha / (1.0 + t / decay)
            l = y_i - (w * x_i + b)
            w = w + a_t * l * (x_i if x_on_w else 1.0)
            b = b + a_t * l * (x_i if x_on_b else 1.0)
            t += 1
            history.append((w, b))
    return history


def _same(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(np.allclose(a, b, rtol=1e-12, atol=1e-9))


# A throwaway problem for the checks: one house of size 2 that sold for 20, so
# the answer is exactly w = 10. The shipped twelve houses are never used here,
# and no check prints the true w or the true b of the shipped data.
CHECK_X, CHECK_Y, CHECK_ANSWER = 2.0, 20.0, 10.0

# A throwaway four house table for the two parameter checks. The numbers are
# invented for the check and have nothing to do with Musterwil.
CHECK_AREAS = np.array([1.0, 2.0, 3.0, 4.0])
CHECK_PRICES = np.array([12.0, 21.0, 29.0, 41.0])


def check_task_1(fixed_step):
    """A one house problem whose answer is exactly w = 10, reached in 10 steps."""
    if _has_blank(fixed_step):
        return fail(BLANK_MESSAGE)
    try:
        history, stopped = fixed_step(CHECK_X, CHECK_Y, w=0.0, alpha=1.0,
                                      eps=0.5, max_steps=50)
    except Exception as exc:
        return fail("The function raised %s: %s" % (type(exc).__name__, exc))
    if len(history) < 2:
        return fail("The loop never took a step. Check that the first "
                    "prediction is wrong by more than eps.")
    if history[0] != 0.0:
        return fail("The first entry of the history should be the starting w.")
    if history[1] < history[0]:
        return fail("The first step went down, from a prediction that was too "
                    "low. Read the miss out loud with its sign, then check "
                    "which branch subtracts alpha.")
    if not stopped:
        return fail("The loop did not reach abs(l) < 0.5 within 50 steps. The "
                    "correct loop arrives in 10 steps of size 1.0.")
    if abs(history[-1] - CHECK_ANSWER) > 1e-9:
        return fail("The loop stopped at w = %r. On this check problem the "
                    "answer is exactly %.1f." % (history[-1], CHECK_ANSWER))
    return ok("Task 1 looks right: 10 steps of size 1.0, landing on w = 10.0.")


def check_task_2(robbins_monro):
    """The same throwaway house. A proportional step should land in a few steps."""
    if _has_blank(robbins_monro):
        return fail(BLANK_MESSAGE)
    try:
        history, stopped = robbins_monro(CHECK_X, CHECK_Y, w=0.0, alpha=0.2,
                                         eps=1e-6, max_steps=200)
    except Exception as exc:
        return fail("The function raised %s: %s" % (type(exc).__name__, exc))
    if len(history) < 2:
        return fail("The loop never took a step.")
    if not np.all(np.isfinite(np.asarray(history, dtype=float))):
        return fail("w became infinite or not a number. The miss is being "
                    "multiplied by (1 - alpha * x) at every step, so it grows "
                    "whenever alpha is above 2 / x.")
    if history[1] < history[0]:
        return fail("w is running away from the answer. The miss is positive "
                    "here, so the update has to add.")
    if not stopped:
        return fail("The loop did not reach abs(l) < 1e-06 within 200 steps. "
                    "The correct loop arrives in 33 steps.")
    if abs(history[-1] - CHECK_ANSWER) > 1e-5:
        return fail("The loop stopped at w = %r. On this check problem the "
                    "answer is %.1f." % (history[-1], CHECK_ANSWER))
    return ok("Task 2 looks right: 33 steps, landing on w = 10.0 to six decimals.")


def check_task_3(rm_alternating):
    """Run the participant function beside three variants and name the match."""
    args = (CHECK_AREAS, CHECK_PRICES, 5.0, 0.0, 0.05, 40)
    if _has_blank(rm_alternating):
        return fail(BLANK_MESSAGE)
    try:
        got = rm_alternating(*args, seed=3)
    except Exception as exc:
        return fail("The function raised %s: %s" % (type(exc).__name__, exc))
    correct = _ref_alternating(*args, seed=3, recompute=True)
    if len(got) != len(correct):
        return fail("The history has %d entries and it should have %d. Append "
                    "one (w, b) pair per step." % (len(got), len(correct)))
    if _same(got, correct):
        return ok("Task 3 looks right: w is updated, the miss is measured "
                  "again, then b is updated.")
    if _same(got, _ref_alternating(*args, seed=3, recompute=False)):
        return fail("You updated w and then reused the old miss for b. Page 31 "
                    "measures the miss a second time, because moving w changed "
                    "the prediction.")
    if _same(got, _ref_alternating(*args, seed=3, recompute=True, x_on_b=True)):
        return fail("The update for b carries a factor x. The base does not "
                    "multiply the area, so its update has no x in it.")
    return fail("The history does not match the alternating rule of page 31. "
                "Check the order: miss, update w, miss again, update b.")


def check_task_4(sgd):
    """Run the participant function beside four variants and name the match."""
    args = (CHECK_AREAS, CHECK_PRICES, 5.0, 0.0, 0.02, 12)
    if _has_blank(sgd):
        return fail(BLANK_MESSAGE)
    try:
        got = sgd(*args, seed=4)
    except Exception as exc:
        return fail("The function raised %s: %s" % (type(exc).__name__, exc))
    correct = _ref_sgd(*args, seed=4)
    if len(got) != len(correct):
        return fail("The history has %d entries and it should have %d. Append "
                    "one (w, b) pair per house visited." % (len(got), len(correct)))
    if _same(got, correct):
        return ok("Task 4 looks right: the slope carries the factor x and the "
                  "base does not.")
    if _same(got, _ref_sgd(*args, seed=4, x_on_w=False)):
        return fail("This is the task 3 rule again. The derivative of the "
                    "squared loss with respect to w carries a factor x.")
    if _same(got, _ref_sgd(*args, seed=4, x_on_b=True)):
        return fail("Both updates carry a factor x. Take the two derivatives "
                    "separately: only the slope multiplies the area.")
    if _same(got, _ref_sgd(*args, seed=4, shuffle=False)):
        return fail("The houses are being visited in area order. Keep the "
                    "shuffle: the table is sorted, so a fixed order ends every "
                    "pass on the largest house.")
    return fail("The history does not match the update taken from the "
                "derivative of the squared loss. Check both lines.")


# ---------------------------------------------------------------- debrief ---
COACH_CONTRACT = """You are coaching a participant of a course for working
professionals. They are filling in a small numerical loop. Follow this contract.
DIAGNOSE: say what their numbers show, in one sentence.
GUIDE: ask one or two targeted questions.
HINT: give a strategy, never a line of code.
IMPLEMENT TOGETHER: if they ask, work on the one line they are stuck on.
Never return a complete drop in function unprompted."""


def _recover_source(name):
    """Return the participant's most recent cell that defines `name`, if any."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None:
            return ""
        history = shell.user_ns.get("In", [])
        for cell in reversed(history):
            if isinstance(cell, str) and ("def %s" % name) in cell:
                return cell
    except Exception:
        return ""
    return ""


def generate_debrief_paste(task_name, numbers, function_name=None):
    """Build a text block the participant pastes into an assistant they use.

    No API key, no client, no network call. The block carries the coaching
    contract, the numbers their loop produced, and their own cell if the
    notebook can recover it.
    """
    lines = [COACH_CONTRACT, "",
             "EXERCISE: hs26-bmai-w1-opt-rm-house, %s." % task_name,
             "The data is twelve fictitious houses. The model is y = w x + b.",
             "The miss is l = y - y_hat and the update adds.",
             "", "WHAT MY LOOP PRODUCED:"]
    for key in sorted(numbers):
        lines.append("  %s = %s" % (key, numbers[key]))
    source = _recover_source(function_name) if function_name else ""
    if source:
        lines += ["", "MY CURRENT CELL:", source]
    lines += ["", "Please start with DIAGNOSE."]
    return "\n".join(lines)
'''

HOUSES_CSV = r'''# Fictitious house data for hs26-bmai-w1-opt-rm-house. Musterwil is an invented municipality, these houses are invented, these prices are invented. Generated by data/generate_houses.py, seed 42. Not real market data.
house_id,area_m2,price_chf
1,101,1083000
2,101,1072000
3,102,1089000
4,116,1201000
5,146,1529000
6,147,1536000
7,158,1623000
8,175,1811000
9,181,1851000
10,191,1913000
11,202,2048000
12,217,2158000
'''

with open("housetools.py", "w") as _fh:
    _fh.write(HOUSETOOLS_PY.lstrip("\n"))
with open("houses.csv", "w") as _fh:
    _fh.write(HOUSES_CSV)

import importlib

import numpy as np

import housetools
H = importlib.reload(housetools)

area, price, x, y = H.load_houses("houses.csv")

print("hs26-bmai-w1-opt-rm-house, notebook %s" % NOTEBOOK_VERSION)
print("")
H.print_houses(area, price, x, y)


---

## Act 0. The house from the lecture

One house. Living area `x = 234` square metres, asking price `y = 198789` francs.
The model is `y = w x`, with a single parameter to find.

Nothing to fill in here. Run the cell and compare the four rows against the board.


In [ ]:
# Act 0. The four guesses the lecturer made this morning, in raw francs and raw
# square metres, so the numbers match the board exactly.
x_lecture, y_lecture = H.LECTURE_X, H.LECTURE_Y

for w in (900.0, 350.0, 850.0, 849.5):
    y_hat = w * x_lecture
    print("w = %8.1f   y_hat = %10.1f   l = y - y_hat = %10.1f"
          % (w, y_hat, y_lecture - y_hat))


### What just happened?

The lecturer found `849.5` by eye. A computer needs a rule.

Discuss with your neighbour. No code required.

- Look at the sign of `l` in each row. What does a positive `l` tell you to do to `w`?
- The third guess misses by 111 francs and the fourth by 6. Which row would you have
  stopped on, and what would you have needed to know in order to decide?
- The four guesses came from a person reading a number off a page. Write the rule
  that person was following, in one sentence of English.


---

## Act 1. The fixed step algorithm

Deck page 30, in the form it had before the lecturer crossed it out. The rule reads
the sign of the miss and steps by a fixed amount `alpha`.

```
guess w
repeat
    y_hat = w x
    l     = y - y_hat
    if l < 0:   w = w - alpha        (the prediction was too high)
    else:       w = w + alpha        (the prediction was too low)
until abs(l) < eps
```

**Notation, fixed for the whole weekend.** `l` is the signed miss `y - y_hat`,
`alpha` is the step size, and `eps` is the stopping tolerance in `abs(l) < eps`. The
step size is always called `alpha`, because the 11:00 gradient descent lecture spends
`eps` on the width of a measuring window.

**Units.** From here on the notebook works in scaled units: `x` is the living area in
units of 100 square metres, so it runs from 1.01 to 2.17, and `y` is the asking price
in thousands of francs. The scaling is the reason `alpha` lands in a range you can
turn by hand.

**Task 1. Fill in the four blanks.**


In [ ]:
# ---------------------------------------------------------------------
# Task 1: the fixed step algorithm, deck page 30, before the crossing out
# ---------------------------------------------------------------------
def fixed_step(x1, y1, w, alpha, eps, max_steps=500):
    """Nudge w by a constant amount, in the direction the miss points."""
    history = [w]
    for step in range(max_steps):
        # COMPLETE THIS LINE: the prediction of the model y = w x
        y_hat = ___
        # COMPLETE THIS LINE: the signed miss, target minus prediction
        l = ___
        if abs(l) < eps:
            return history, True          # the loop stopped on its own
        if l < 0:
            # COMPLETE THIS LINE: the prediction was too high
            w = ___
        else:
            # COMPLETE THIS LINE: the prediction was too low
            w = ___
        history.append(w)
    return history, False                 # the loop ran out of steps


print("fixed_step is defined. Run the next cell to check it.")


In [ ]:
# The check runs your loop on a throwaway house of size 2 that sold for 20, so the
# answer there is exactly w = 10. The twelve Musterwil houses are left alone.
# Nothing below the check runs until the check passes.
if H.check_task_1(fixed_step):

    x1, y1 = x[0], y[0]
    print("")
    print("house 1 of the table: x = %.2f, y = %.1f" % (x1, y1))

    hist, stopped = fixed_step(x1, y1, w=900.0, alpha=10.0, eps=1.0, max_steps=400)
    print("stopped on its own: %s   steps taken: %d" % (stopped, len(hist) - 1))
    print("last six values of w: %s" % [round(v, 2) for v in hist[-6:]])
    print("last six misses     : %s" % [round(y1 - v * x1, 2) for v in hist[-6:]])

    # the same loop again, with the tolerance widened from 1 franc to 12
    hist2, stopped2 = fixed_step(x1, y1, w=900.0, alpha=10.0, eps=12.0, max_steps=400)
    print("")
    print("with eps = 12.0")
    print("stopped on its own: %s   steps taken: %d   final w = %.1f   final miss = %.1f"
          % (stopped2, len(hist2) - 1, hist2[-1], y1 - hist2[-1] * x1))
    print("")
    print("the exact answer to this one house equation is y / x = %.4f" % (y1 / x1))


In [ ]:
# The picture of the same run. The dashed line is the exact answer.
H.plot_orbit(hist, x1, y1, title="Fixed step, alpha = 10");


### What just happened?

The loop ran four hundred steps and stopped on none of them.

Discuss. No code required.

- The last six values of `w` are `1070, 1080, 1070, 1080, 1070, 1080`. Neither of
  those two numbers is the answer `1072.2772`. Why can the loop never reach it?
- The width of the orbit is `alpha` in `w`. What is it in francs?
- Widening `eps` to 12 made the loop stop after 17 steps. Say in one sentence why
  that is a hollow kind of stopping.
- Two knobs could make this loop stop: a smaller `alpha` or a larger `eps`. Neither
  one is convergence. What would have to change about the *rule* instead?


---

## Act 2. The update the lecturer boxed

On page 30 the whole `if / else` block is crossed out with a large X, and one line
is boxed in its place.

```
w <- w + alpha * l
```

The step is now proportional to the miss. Nothing else changes, and `alpha` still
never changes.

> **The sign convention, said out loud.** The genie exercise forty minutes ago wrote
> `loss = Q - target` and updated `Q := Q - alpha * loss`. This notebook follows the
> BMAI deck and writes `l = y - y_hat` with the update `w <- w + alpha * l`. Both
> rules move the estimate toward the target. The two exercises define the miss with
> opposite signs, and that is deliberate rather than a slip.

**Task 2. Fill in the one blank.**


In [ ]:
# ---------------------------------------------------------------------
# Task 2: the update the lecturer boxed, deck page 30, after the crossing out
# ---------------------------------------------------------------------
def robbins_monro(x1, y1, w, alpha, eps, max_steps=200):
    """Move by an amount proportional to the miss."""
    history = [w]
    for step in range(max_steps):
        y_hat = w * x1
        l = y1 - y_hat
        if abs(l) < eps:
            return history, True
        # COMPLETE THIS LINE: the boxed update from the lecture
        w = ___
        history.append(w)
    return history, False


print("robbins_monro is defined. Run the next cell to check it.")


In [ ]:
if H.check_task_2(robbins_monro):

    print("")
    hist_rm, stopped_rm = robbins_monro(x1, y1, w=900.0, alpha=0.5, eps=1.0)
    print("alpha = 0.5")
    print("stopped on its own: %s   steps taken: %d   final w = %.4f"
          % (stopped_rm, len(hist_rm) - 1, hist_rm[-1]))
    print("trace: %s" % [round(v, 2) for v in hist_rm])
    print("")
    print("Act 1 took 400 steps and never stopped.")

    # what happens when the step size gets large
    print("")
    print("the same loop at three step sizes, 60 steps each:")
    for a in (0.5, 1.9, 2.2):
        h, s = robbins_monro(x1, y1, w=900.0, alpha=a, eps=1.0, max_steps=60)
        print("  alpha = %.1f   final w = %16.4f   stopped: %s" % (a, h[-1], s))

    print("")
    print("every step multiplies the miss by (1 - alpha * x), so the loop converges")
    print("while alpha < 2 / x = %.4f" % (2.0 / x1))


In [ ]:
H.plot_orbit(hist_rm, x1, y1, title="Robbins-Monro, alpha = 0.5");


### What just happened?

Eight steps, against four hundred that never arrived.

Discuss. No code required.

- `alpha` did not change during the run. What shrank instead, and why does that make
  the steps get smaller on their own?
- At `alpha = 2.2` the estimate reached about minus 29 million. Work out
  `1 - alpha * x` at `alpha = 2.2` and `x = 1.01`, and say what that number does to
  the miss at every step.
- The threshold `alpha < 2 / x` is worth writing down. Compute it for the largest
  house in the table, `x = 2.17`. Is `alpha = 0.5` safe for every house?


---

## Act 3. Two parameters, twelve houses

A price of zero for a house of zero area is a poor model, so the deck adds a base on
page 31.

```
y = w x + b            w is the price per unit of area, b is the base
```

Page 31 updates the two parameters one after the other, and it writes the miss out a
second time in between.

```
draw one house (x_i, y_i)
l = y_i - (w x_i + b)
w <- w + alpha * l
l = y_i - (w x_i + b)        <-- measured again, because w just moved
b <- b + alpha * l
```

That second measurement is the entire content of the word "alternating", and it is
the line participants drop.

**Task 3. Fill in the three blanks.**


In [ ]:
# ---------------------------------------------------------------------
# Task 3: two parameters, alternating, deck page 31
# ---------------------------------------------------------------------
def rm_alternating(areas, prices, w, b, alpha, n_steps, seed=0):
    """One house at a time, drawn at random. Update w, then update b."""
    rng = np.random.default_rng(seed)
    history = [(w, b)]
    for step in range(n_steps):
        i = int(rng.integers(len(areas)))
        x_i, y_i = areas[i], prices[i]
        l = y_i - (w * x_i + b)
        # COMPLETE THIS LINE: update the slope
        w = ___
        # COMPLETE THIS LINE: measure the miss again, with the new w
        l = ___
        # COMPLETE THIS LINE: update the base
        b = ___
        history.append((w, b))
    return history


print("rm_alternating is defined. Run the next cell to check it.")


In [ ]:
# The collapse plan. If the block runs short the TA sets PREFILL_TASK3 = True in
# the setup cell, and this cell hands you a finished loop so that act 4 still fits
# in the hour. With the flag left at False this cell does nothing.
if PREFILL_TASK3:
    def rm_alternating(areas, prices, w, b, alpha, n_steps, seed=0):
        """One house at a time, drawn at random. Update w, then update b."""
        return H._ref_alternating(areas, prices, w, b, alpha, n_steps, seed=seed)
    print("Task 3 was prefilled by the TA. Read the loop above, then move on.")
else:
    print("PREFILL_TASK3 is False, so your own loop is the one being used.")


In [ ]:
if H.check_task_3(rm_alternating):

    print("")
    hist3 = rm_alternating(x, y, w=500.0, b=0.0, alpha=0.1, n_steps=3600, seed=1)
    w3, b3 = hist3[-1]
    print("after 3600 steps from (w, b) = (500, 0) at alpha = 0.1")
    print("  w = %.4f   b = %.4f   mean squared error = %.4f"
          % (w3, b3, H.mse(w3, b3, x, y)))

    hist3_long = rm_alternating(x, y, w=500.0, b=0.0, alpha=0.1, n_steps=20000, seed=1)
    w3l, b3l = hist3_long[-1]
    print("")
    print("running it much longer does not tidy it up. After 20000 steps")
    print("  w = %.4f   b = %.4f   mean squared error = %.4f"
          % (w3l, b3l, H.mse(w3l, b3l, x, y)))


In [ ]:
H.plot_fit_grid(hist3, x, y, picks=(0, 1, 2, 5, 20, -1));


In [ ]:
H.plot_loss_curve(hist3, x, y);


### What just happened?

The line fits. It gets close. It settles into a neighbourhood and then rattles
around inside it forever.

Discuss. No code required.

- Twenty thousand steps did no better than thirty six hundred. What is the loop
  doing with all those extra steps?
- Each step is aimed at one house. Once the line is roughly right, what does a step
  aimed at house 1 do to the fit on house 12?
- Look at the loss curve. Name the point on it where more steps stop buying anything.
- The dashed line on the loss curve is the best line that exists for this data. The
  loop is above it. What would have to change to close the gap?


---

## Act 4. The naming

Deck page 35 and page 36 pick the squared error as the loss for regression. Write it
for one house.

```
L(w, b) = (1/2) (y - w x - b)^2
```

Differentiate it once with respect to each parameter.

```
dL/dw = -(y - w x - b) * x = -l * x
dL/db = -(y - w x - b)     = -l
```

Moving downhill in the loss means stepping against the derivative, which gives

```
w <- w + alpha * l * x
b <- b + alpha * l
```

Compare the pair against act 3. The base update is identical. The slope update has
picked up a factor `x`.

**Task 4. Fill in the two blanks.**


In [ ]:
# ---------------------------------------------------------------------
# Task 4: the same loop, with the derivative of the squared loss
# ---------------------------------------------------------------------
def sgd(areas, prices, w, b, alpha, n_epochs, seed=0, decay=None):
    """One house at a time, both parameters moved down the squared loss."""
    rng = np.random.default_rng(seed)
    history = [(w, b)]
    t = 0
    for epoch in range(n_epochs):
        for i in rng.permutation(len(areas)):     # a fresh order on every pass
            x_i, y_i = areas[i], prices[i]
            a_t = alpha if decay is None else alpha / (1.0 + t / decay)
            l = y_i - (w * x_i + b)
            # COMPLETE THIS LINE: the slope, moved down the squared loss
            w = ___
            # COMPLETE THIS LINE: the base, moved down the squared loss
            b = ___
            t += 1
            history.append((w, b))
    return history


print("sgd is defined. Run the next cell to check it.")


In [ ]:
if H.check_task_4(sgd):

    w_lsq, b_lsq = H.least_squares(x, y)

    h4 = sgd(x, y, w=500.0, b=0.0, alpha=0.05, n_epochs=300, seed=1)
    h5 = sgd(x, y, w=500.0, b=0.0, alpha=0.3, n_epochs=300, seed=1, decay=200.0)
    h6 = sgd(x, y, w=500.0, b=0.0, alpha=0.3, n_epochs=3000, seed=1, decay=200.0)

    rows = [
        ("act 3 rule, alpha = 0.1, 3600 steps", w3, b3),
        ("act 4 rule, alpha = 0.05, 300 passes", h4[-1][0], h4[-1][1]),
        ("act 4 rule, alpha_t = 0.3/(1 + t/200), 300 passes", h5[-1][0], h5[-1][1]),
        ("act 4 rule, the same shrinking step, 3000 passes", h6[-1][0], h6[-1][1]),
        ("the best line that exists for this data", w_lsq, b_lsq),
    ]
    print("")
    print("%-52s %10s %10s %12s" % ("", "w", "b", "mse"))
    for label, wv, bv in rows:
        print("%-52s %10.4f %10.4f %12.4f" % (label, wv, bv, H.mse(wv, bv, x, y)))


In [ ]:
# The data was generated from a line. Here it is, revealed now that you have
# found it for yourself. Musterwil is invented and so is this line.
print("the line the generator used:   w = 950.0   b = 120.0")
print("the best line for the sample:  w = %.4f   b = %.4f" % (w_lsq, b_lsq))
print("")
print("The two disagree, and that is correct. The generator added noise to every")
print("price, so the twelve houses you were given are not exactly on the line they")
print("came from. Your loop can only find the best line for the sample in front of")
print("it. Saturday morning is about that gap.")


In [ ]:
H.plot_wb_path(h4, x, y, every=5);


### The naming

Deck page 56 puts 1982 beside today in a table. Every row changes except the first,
and the last row reads:

| | 1982 | Today |
|---|---|---|
| Training algorithm | Robbins-Monro algorithm | Stochastic gradient descent |

Page 57 and page 59 name the upgrades that compose into that equality. Quoted from
the deck, with the typographic quotation marks transcribed as ASCII:

> The Robbins-Monro update rule uses only the difference between target and
> estimation. A more efficient update rule uses the gradient of the loss function.

> At each iteration of the update loop, don't recompute the loss on the entire data;
> use only a "batch" (i.e., a subset of the dataset) ... Integrating all these
> improvements yields stochastic gradient descent, one of the most powerful modern
> algorithms for estimating dependencies on data.

And the honest algebra, because "Robbins-Monro is stochastic gradient descent" is
true in a precise sense and sloppy in a loose one.

```
base:    Robbins-Monro says    b <- b + alpha * l
         the derivative says   b <- b + alpha * l
         the same line.

slope:   Robbins-Monro says    w <- w + alpha * l
         the derivative says   w <- w + alpha * l * x
         the same line once the step size for w is read as alpha / x.
```

So for the base, the rule the lecturer boxed already **is** stochastic gradient
descent on the squared loss. For the slope it is the same rule with the feature
factor folded into the step size, which is why it converges and also why it settles
slightly off the best line on data where the areas differ. Put the factor back, draw
one house at a time, and you have written the right-hand column of the table. The
name was the only new thing.


---

## Extensions

Everything above is the exercise. Everything below is for whoever has time left.


### Extension A. The batch rule on page 32, and why it stalls

Page 32 replaces the single house with the whole table and writes the miss as the
mean of the signed residuals.

```
l = (1/n) sum over houses of (y_i - (w x_i + b))
w <- w + alpha * l
l = (1/n) sum over houses of (y_i - (w x_i + b))
b <- b + alpha * l
```

Run it from three different starting points and read the last column.


In [ ]:
def batch_deck(areas, prices, w, b, alpha, n_passes):
    """Deck page 32, exactly as written: l is the mean signed residual."""
    for _ in range(n_passes):
        l = float(np.mean(prices - (w * areas + b)))
        w = w + alpha * l
        l = float(np.mean(prices - (w * areas + b)))
        b = b + alpha * l
    return w, b


print("%-22s %10s %10s %12s %18s"
      % ("starting point", "w", "b", "mse", "w*mean(x) + b"))
for start in ((500.0, 0.0), (1200.0, 0.0), (950.0, 120.0)):
    wb, bb = batch_deck(x, y, start[0], start[1], 0.05, 20000)
    print("%-22s %10.4f %10.4f %12.4f %18.4f"
          % (str(start), wb, bb, H.mse(wb, bb, x, y), wb * x.mean() + bb))

print("")
print("mean(y) = %.4f" % y.mean())


### What just happened in extension A?

Three starting points, three different answers, and every one of them satisfies
`w * mean(x) + b = mean(y)`.

The mean signed residual is one number. Setting it to zero is one equation, and the
model has two unknowns, so the rule pins down the line through the centre of mass of
the data and leaves the slope to whatever it happened to be. The per-house rule of
act 3 escapes this because each house pushes in a slightly different direction, and
averaging over the table throws that information away.

This is the honest answer to "the slide says mean, so why did the notebook not do
that". The required path takes the per-house route, and page 59 backs it: using a
subset of the data at each iteration is exactly the upgrade that produces stochastic
gradient descent.


### Extension B. The shrinking step, and the classical conditions

Act 4 already accepts a `decay` argument. With `decay=None` the step size is a
constant and the iterates rattle inside a neighbourhood forever. With a decay the
step shrinks as `alpha_t = alpha / (1 + t / decay)` and they settle.

The classical conditions on a Robbins-Monro step size come from the AINIT
reinforcement learning deck, because the BMAI deck never writes an indexed step
size at all. The steps must add up without bound, so the estimate can travel from
any starting point to anywhere it needs to go, and their squares must add up to
something finite, so the noise cancels in the end. The sequence `alpha_t = 1/t`
satisfies both.


In [ ]:
def band(history, tail):
    """How wide a box the last `tail` iterates rattle inside."""
    h = np.asarray(history[-tail:], dtype=float)
    return float(h[:, 0].max() - h[:, 0].min()), float(h[:, 1].max() - h[:, 1].min())


if H.filled_in(sgd):
    h_const = sgd(x, y, w=500.0, b=0.0, alpha=0.05, n_epochs=3000, seed=1)
    h_decay = sgd(x, y, w=500.0, b=0.0, alpha=0.3, n_epochs=3000, seed=1, decay=200.0)

    for label, h in (("constant step", h_const), ("shrinking step", h_decay)):
        bw, bb = band(h, 1200)
        print("%-16s final w = %.4f  b = %.4f  mse = %.6f"
              % (label, h[-1][0], h[-1][1], H.mse(h[-1][0], h[-1][1], x, y)))
        print("%-16s over the last 1200 steps, w moved across %.4f and b across %.4f"
              % ("", bw, bb))

    print("")
    print("the best line for this data: w = %.4f  b = %.4f  mse = %.6f"
          % (w_lsq, b_lsq, H.mse(w_lsq, b_lsq, x, y)))
else:
    print("Task 4 still has a blank in it, so this extension has nothing to run.")


### Extension C. A second feature

Deck pages 33 and 34 take `x_i` to be a pair, area and rooms, and the model becomes

```
y = w_1 x_1 + w_2 x_2 + b
```

Writing the update as a loop over the coordinates is Carlos's stage 4 and stage 5
for the 11:00 lecture, and it is the bridge into the grasshopper block after lunch,
where each coordinate is handled by its own grasshopper.

This one is a stretch, and it is honestly unfinished. There is no stub, no check and
no second column: `data/generate_houses.py` produces area and price only, so a room
count would have to be added to the generator first. If you want to try it now,
invent a second feature from the area you already have, fit
`y = w_1 x_1 + w_2 x_2 + b` with three copies of the act 4 update, and watch what
happens when the two features are strongly related to each other.


---

## If you are stuck, or if you want to go deeper

The cell below builds a block of text carrying the coaching contract, whatever your
loop produced, and your own current cell if the notebook can recover it. Copy it and
paste it into whichever assistant you already have open. No key and no account are
needed for this notebook, and nothing is sent anywhere by the notebook itself.


In [ ]:
# @title Build a paste block for an assistant of your choice.
# This cell works whether or not your loops ran, because a participant who is
# stuck is exactly the one who needs it.
if "h4" in globals():
    numbers = {"final w": round(h4[-1][0], 4),
               "final b": round(h4[-1][1], 4),
               "mean squared error": round(H.mse(h4[-1][0], h4[-1][1], x, y), 4),
               "best possible mse": round(H.mse(w_lsq, b_lsq, x, y), 4)}
else:
    numbers = {"status": "task 4 has not produced any numbers yet"}

paste = H.generate_debrief_paste(
    task_name="task 4, stochastic gradient descent on twelve houses",
    numbers=numbers,
    function_name="sgd")
print(paste)


---

## What comes next

At 11:00 the lecture replaces the miss `l` with the slope of the loss, measured over
a small window and then in the limit as the window closes. The update keeps the shape
you wrote in act 4.

At 12:00 the grasshopper game gives you the same rule with one grasshopper per
coordinate.

On Saturday morning the question becomes the one act 4 raised at the end: your loop
found the best line for the twelve houses in front of it, and the generator used a
different line. Telling those two apart is validation.

**A last reminder.** Musterwil is invented, the twelve houses are invented, and the
prices are invented. The generator that made them is committed beside this notebook
at `data/generate_houses.py`, seed 42.
